In [ ]:
!pip install kagglehub

In [ ]:
import kagglehub

path = kagglehub.dataset_download("orvile/english-to-turkish-sentence-pairs") #https://www.kaggle.com/datasets/orvile/english-to-turkish-sentence-pairs/data

print("Path to dataset files:", path)

100%|██████████| 22.0M/22.0M [00:02<00:00, 9.56MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/orvile/english-to-turkish-sentence-pairs/versions/1


In [ ]:
import pandas as pd
import os

print(os.listdir(path))  # see available files


file_path = path + "/Sentence pairs in English-Turkish - 2025-05-06.tsv"

df = pd.read_csv(file_path, sep="\t", engine="python", on_bad_lines="skip")

df.head()

['Sentence pairs in English-Turkish - 2025-05-06.tsv']


,1276,Let's try something.,477649,Bir şeyler deneyelim!
0,1277,I have to go to sleep.,1166752,Yatmaya gitmek zorundayım.
1,1277,I have to go to sleep.,1283434,Uyumam lazım.
2,1280,Today is June 18th and it is Muiriel's birthday!,1090572,Bugün 18 Haziran ve Muiriel'in doğum günü!
3,1282,Muiriel is 20 now.,475258,Muiriel şimdi 20 yaşında.
4,1282,Muiriel is 20 now.,1090575,Muiriel şimdi 20.


In [ ]:
pronouns = [" I ", " You ", " He ", " She ", " They ", " We "]

df = df[df["Let's try something."].str.contains("|".join(pronouns))]

In [ ]:
len(df)

38103

In [ ]:
df["word_count"] = df["Let's try something."].str.split().str.len()

df = df[df["word_count"] >= 4]

In [ ]:
len(df)

37986

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 70.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

keywords = ["but", "though", "yet", "while", "whereas", "however", "although", "since", "if","unless", "before", "after", "when", "once", ";"]
clause_deps = ["ccomp", "xcomp", "advcl", "acl", "relcl"]

filtered_sentences = []

for sentence in df["Let's try something."]:
    doc = nlp(sentence)

    has_keyword = any(token.text.lower() in keywords for token in doc)
    has_clause = any(token.dep_ in clause_deps for token in doc)

    # OR condition
    if has_keyword or has_clause:
        filtered_sentences.append(sentence)

print("Filtered sentence count:", len(filtered_sentences))

Filtered sentence count: 28694


In [ ]:
df.head(20)

,1276,Let's try something.,477649,Bir şeyler deneyelim!,word_count
15,1292,I don't know if I have the time.,1090626,Zamanım olup olmadığını bilmiyorum.,8
17,1295,You're in better shape than I am.,1090630,Siz benden daha çok formdasınız.,7
28,1309,I'll call them tomorrow when I come back.,1522115,Yarın geri döndüğümde onları arayacağım.,8
29,1309,I'll call them tomorrow when I come back.,2581164,Geri döndüğümde onları yarın ararım.,8
39,1318,The last person I told my idea to thought I wa...,2581159,Fikrimi söylediğim son kişi deli olduğumu düşü...,12
40,1319,"If the world weren't in the shape it is now, I...",1117336,"Dünya şimdi olduğu durumda olmasa, kimseye güv...",14
41,1319,"If the world weren't in the shape it is now, I...",3007235,Eğer dünya şimdiki şeklinde olmasaydı herhangi...,14
54,1332,For some reason I feel more alive at night.,1117367,Bazı sebeplerden dolayı geceleri daha canlı hi...,9
61,1337,"When I grow up, I want to be a king.",2736196,Büyüyünce bir kral olmak istiyorum.,10
78,1355,"I may be antisocial, but it doesn't mean I don...",2438201,"Asosyal olabilirim , ama bu insanlarla konuşma...",13


In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

col = "Let's try something."

keywords = ["but", "though", "yet", "while", "whereas", "however",
            "although", "since", "if", "unless", "before",
            "after", "when", "once", ";"]

clause_deps = ["ccomp", "xcomp", "advcl", "acl", "relcl"]

filtered_rows = []

for idx, sentence in df[col].items():

    if not isinstance(sentence, str):
        continue

    # Remove questions
    if sentence.strip().endswith("?"):
        continue

    doc = nlp(sentence)

    has_keyword = any(token.text.lower() in keywords for token in doc)
    has_clause = any(token.dep_ in clause_deps for token in doc)

    if has_keyword or has_clause:
        filtered_rows.append(df.loc[idx])

filtered_df = pd.DataFrame(filtered_rows)

print("Final filtered size:", len(filtered_df))

Final filtered size: 25465


In [ ]:
filtered_df.to_csv("data.csv", index=False)